# 06 — Social (publication charts)

The owner reviewed `04-viz` and confirmed the lineup, so this notebook renders the
publication-ready set. Each chart is produced in up to **two targets** from the same data:
- **Social** — full chrome (title, subtitle, source, `@unwelcomedata` watermark),
  `twitter_landscape` (1600×900) → `outputs/social/`. Optimized for Bluesky/X.
- **Web** — `web_mode=True` (drops title/subtitle/source, keeps only the watermark),
  `web` preset (1664×936) → `outputs/web/`. Embedded on the Pages site, which supplies
  its own headings.

**The lineup (confirmed in `04-viz`):**
1. **SOTU self vs collective, by president** (line) — social + web.
2. **SOTU “I” presidents vs “we” presidents** (side-by-side) — social + web.
3. **Inaugural “I” vs “we” presidents** (side-by-side) — social + web.
4. **Every SOTU president’s lean, chronological** (diverging, tall) — **web only**.
5. **SOTU average length by president** (ranked bars, tall) — **web only**.
6. **Speeches in the corpus per year in office** (ranked bars, tall) — **web only**.
7. **Applause & laughter per president** (stacked bars + footnote) — social + web.
8. **Every president’s most-used word, 1789→present** (ranked bars, colored) — social (taller 1600×1080) + web.
- **Word clouds** — Lincoln (distinctive), FDR (most-used), Reagan (phrases) — **social only**.

Titles are **descriptive** (house default), not conclusion headlines — present the data,
let it carry the point. Read-only DuckDB for rendering; a short writable pass rebuilds the
`chart_*` aggregation tables first. Connection closed in the Cleanup cell.

## ⚠️ Rendering note
The project `.venv` is Py3.14 (matplotlib RecursionError) and lacks `wordcloud`. Run this
notebook on the **anaconda `data_projects`** kernel, which has `wordcloud`; all charts use
the shared Pillow factory regardless.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent.parent / 'shared'))

# Word clouds need the `wordcloud` package. Fail with a clear message if the active
# kernel's environment is missing it, rather than a deep import error mid-notebook.
try:
    import wordcloud  # noqa: F401
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "'wordcloud' is not installed in this kernel's environment. "
        "Run this notebook on the anaconda 'data_projects' kernel (pip install wordcloud)."
    )

import duckdb, pandas as pd
from IPython.display import display, Image as IPImage
from chart_factory import _CHART_TYPES
from colors import c
from viz import PRESETS

DB = 'data/project.duckdb'
SRC = 'Miller Center (UVA) speech archive'
soc_w, soc_h, _ = PRESETS['twitter_landscape']   # 1600x900
web_w, web_h, _ = PRESETS['web']                 # 1664x936
social_out = Path('outputs/social'); social_out.mkdir(parents=True, exist_ok=True)
web_out    = Path('outputs/web');    web_out.mkdir(parents=True, exist_ok=True)
print('paths ready — social:', social_out, '| web:', web_out)

## Build the chart aggregation tables

Reuse the exact SQL settled in `04-viz` so the social renders come from the same data as
the approved exploration (no drift). Writable connection for the builds, then reconnect
read-only to render.

In [ ]:
w = duckdb.connect(DB)

# Chart 1: one row per president (avg of their SOTUs) at their first SOTU year.
w.execute('DROP TABLE IF EXISTS chart_sotu_by_president')
w.execute("""CREATE TABLE chart_sotu_by_president AS
  SELECT president, MIN(year) AS first_year, COUNT(*) AS n_sotu,
         ROUND(AVG(self_per_1k),2) self_1k, ROUND(AVG(collective_per_1k),2) coll_1k,
         ROUND(AVG(self_share),3) self_share
  FROM speeches_features WHERE is_sotu_series GROUP BY 1 ORDER BY first_year""")

# self_share per president, per lens (feeds the top-I / top-we side-by-side tables).
w.execute('DROP TABLE IF EXISTS chart_sotu_self_share')
w.execute("""CREATE TABLE chart_sotu_self_share AS
  SELECT president, COUNT(*) n_sotu, ROUND(AVG(self_share),3) self_share
  FROM speeches_features WHERE is_sotu_series GROUP BY 1 HAVING COUNT(*)>=2
  ORDER BY self_share DESC""")
w.execute('DROP TABLE IF EXISTS chart_inaug_self_share')
w.execute("""CREATE TABLE chart_inaug_self_share AS
  SELECT president, ROUND(AVG(self_share),3) self_share
  FROM speeches_features WHERE speech_type='Inaugural Address' GROUP BY 1
  ORDER BY self_share DESC""")

# Chart 2: SOTU top-10 most "I" and top-10 most "we", each labeled in its own units.
w.execute('DROP TABLE IF EXISTS chart_sotu_top_i')
w.execute("""CREATE TABLE chart_sotu_top_i AS
  SELECT president, ROUND(self_share*100,0) AS pct, printf('%.0f%%', self_share*100) AS label
  FROM chart_sotu_self_share ORDER BY self_share DESC LIMIT 10""")
w.execute('DROP TABLE IF EXISTS chart_sotu_top_we')
w.execute("""CREATE TABLE chart_sotu_top_we AS
  SELECT president, ROUND((1-self_share)*100,0) AS pct, printf('%.0f%%', (1-self_share)*100) AS label
  FROM chart_sotu_self_share ORDER BY self_share ASC LIMIT 10""")

# Chart 3: inaugural top-10 "I" / top-10 "we" (mirror of Chart 2, inaugural lens).
w.execute('DROP TABLE IF EXISTS chart_inaug_top_i')
w.execute("""CREATE TABLE chart_inaug_top_i AS
  SELECT president, ROUND(self_share*100,0) AS pct, printf('%.0f%%', self_share*100) AS label
  FROM chart_inaug_self_share ORDER BY self_share DESC LIMIT 10""")
w.execute('DROP TABLE IF EXISTS chart_inaug_top_we')
w.execute("""CREATE TABLE chart_inaug_top_we AS
  SELECT president, ROUND((1-self_share)*100,0) AS pct, printf('%.0f%%', (1-self_share)*100) AS label
  FROM chart_inaug_self_share ORDER BY self_share ASC LIMIT 10""")

# Chart 4 (WEB ONLY): every SOTU president, chronological, lean = self_share-0.5.
w.execute('DROP TABLE IF EXISTS chart_sotu_lean_all')
w.execute("""CREATE TABLE chart_sotu_lean_all AS
  SELECT president, MIN(year) AS fy, ROUND(AVG(self_share)-0.5,3) AS lean,
         CASE WHEN AVG(self_share) >= 0.5
              THEN printf('%.0f%%', AVG(self_share)*100)
              ELSE printf('%.0f%%', (1-AVG(self_share))*100) END AS label
  FROM speeches_features WHERE is_sotu_series GROUP BY president HAVING COUNT(*)>=2
  ORDER BY fy""")

# Chart 5 (WEB ONLY): avg SOTU length by president.
w.execute('DROP TABLE IF EXISTS chart_sotu_length')
w.execute("""CREATE TABLE chart_sotu_length AS
  SELECT president, COUNT(*) n, CAST(ROUND(AVG(word_count),0) AS INTEGER) avg_words,
         printf('%,d', CAST(ROUND(AVG(word_count),0) AS INTEGER)) AS label
  FROM speeches_features WHERE is_sotu_series GROUP BY 1 HAVING COUNT(*)>=2
  ORDER BY avg_words DESC""")

# Chart 6 (WEB ONLY): corpus-coverage rate (speeches per year in office).
w.execute('DROP TABLE IF EXISTS chart_speeches_per_year')
w.execute("""CREATE TABLE chart_speeches_per_year AS
  SELECT president, speeches_per_year, printf('%.1f/yr', speeches_per_year) AS label
  FROM president_terms WHERE reliable_rate ORDER BY speeches_per_year DESC""")

# Chart 7: audience-reaction markers (applause-family + laughter-family), >=20 reactions.
w.execute('DROP TABLE IF EXISTS chart_stage_directions')
w.execute("""CREATE TABLE chart_stage_directions AS
  SELECT president,
         (applause + applauding) AS applause,
         (laughter + laughs)     AS laughter,
         (applause + applauding + laughter + laughs + cheers + booing) AS reaction
  FROM stage_directions_by_president
  WHERE (applause + applauding + laughter + laughs + cheers + booing) >= 20
  ORDER BY reaction DESC""")

# Chart 8: each president's #1 most-used word, chronological, colored by word-family.
TEAL, GOLD, RUST, GRAY = c('teal'), c('gold'), c('spice'), c('gray')
w.execute('DROP TABLE IF EXISTS chart_top_word_timeline')
w.execute(f"""CREATE TABLE chart_top_word_timeline AS
  WITH ranked AS (
    SELECT wf.president, wf.word, wf.count,
      ROW_NUMBER() OVER (PARTITION BY wf.president ORDER BY wf.count DESC) rn,
      MIN(s.year) OVER (PARTITION BY wf.president) fy
    FROM word_freq_by_president wf JOIN speeches_clean s ON s.president=wf.president)
  SELECT fy || '  ·  ' || president AS row_label, word AS top_word, count, fy,
    CASE WHEN word IN ('state','states','union') THEN '{TEAL}'
         WHEN word IN ('world','peace') THEN '{GOLD}'
         WHEN word IN ('people','country') THEN '{RUST}'
         ELSE '{GRAY}' END AS color,
    word AS label
  FROM ranked WHERE rn=1 ORDER BY fy""")

for t in ['chart_sotu_by_president','chart_sotu_self_share','chart_inaug_self_share',
          'chart_sotu_top_i','chart_sotu_top_we','chart_inaug_top_i','chart_inaug_top_we',
          'chart_sotu_lean_all','chart_sotu_length','chart_speeches_per_year',
          'chart_stage_directions','chart_top_word_timeline']:
    print(t, w.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0], 'rows')
w.close()
con = duckdb.connect(DB, read_only=True)
print('reconnected read-only')

In [ ]:
# Dual-render helper: build the SOCIAL image (full chrome) and the WEB image (web_mode,
# no title/subtitle/source), save each to its own dir at exact preset dims, display the
# social inline. Reuses the shared factory's config->template routing so social and web
# come from identical configs and can't drift.
def render_targets(num, name, base, *, social=True, web=True,
                   social_img_height=None, web_img_height=None):
    builder = _CHART_TYPES[base['type']]
    stem = f'{num:02d}_{name}'
    if social:
        cfg = {**base, 'preset': 'twitter_landscape', 'web_mode': False}
        if social_img_height is not None:
            cfg['img_height'] = social_img_height
        img = builder(cfg, con)
        p = social_out / f'{stem}.png'
        img.save(p, format='PNG', optimize=True)
        print(f'social → {p}  ({img.size[0]}x{img.size[1]})')
        display(img)
    if web:
        cfg = {**base, 'preset': 'web', 'web_mode': True,
               'title': '', 'subtitle': None, 'source': None}
        if web_img_height is not None:
            cfg['img_height'] = web_img_height
        imgw = builder(cfg, con)
        pw = web_out / f'{stem}.png'
        imgw.save(pw, format='PNG', optimize=True)
        print(f'web    → {pw}  ({imgw.size[0]}x{imgw.size[1]})')

def render_social_only(num, name, base):
    """Word clouds: social target only (they live in the picker on the web side)."""
    builder = _CHART_TYPES[base['type']]
    stem = f'{num:02d}_{name}'
    img = builder({**base, 'preset': 'twitter_landscape', 'web_mode': False}, con)
    p = social_out / f'{stem}.png'
    img.save(p, format='PNG', optimize=True)
    print(f'social → {p}  ({img.size[0]}x{img.size[1]})')
    display(img)

print('helpers ready')

## Chart 1 — State of the Union: self vs collective, by president (social + web)

Each dot = one president’s average across their SOTUs, placed at their first SOTU year;
two connected lines (collective we/us/our vs self I/me/my) per 1,000 words. Ticks sit
only at real presidents’ first years.

In [ ]:
render_targets(1, 'sotu_self_vs_collective_by_president', {
    'type': 'line', 'table': 'chart_sotu_by_president', 'x_col': 'first_year',
    'series': [{'col': 'coll_1k', 'label': 'collective (we/us/our)', 'color': c('teal')},
               {'col': 'self_1k', 'label': 'self (I/me/my)', 'color': c('spice')}],
    'connect': True, 'markers': True, 'legend': True, 'label_last': False, 'y_min': 0,
    'x_axis_label': '', 'y_axis_label': 'Pronouns per 1,000 words',
    'x_ticks': con.execute('SELECT first_year FROM chart_sotu_by_president ORDER BY first_year').df()['first_year'].tolist(),
    'x_fmt': (lambda v: f'{int(v)}'),
    'title': 'State of the Union — self vs collective, by president',
    'subtitle': 'Each dot = one president\u2019s average across their SOTUs, placed at their first SOTU year. Per 1,000 words.',
    'source': SRC,
})

## Chart 2 — SOTU: the “I” presidents and the “we” presidents (social + web)

Two panels, each in its own units: left = the 10 presidents whose SOTU first-person
pronouns are most often “I” (% I); right = the 10 most “we” (% we).

In [ ]:
render_targets(2, 'sotu_i_vs_we_presidents', {
    'type': 'side_by_side_bars',
    'table_left': 'chart_sotu_top_i', 'table_right': 'chart_sotu_top_we',
    'category_col': 'president', 'value_col': 'pct', 'label_col': 'label',
    'left_title': 'Most “I”', 'right_title': 'Most “we”',
    'left_color': c('spice'), 'right_color': c('teal'),
    'title': 'State of the Union — the “I” presidents and the “we” presidents',
    'subtitle': 'Left: share of first-person pronouns that are “I”. Right: share that are “we”. Top 10 each (≥2 SOTUs).',
    'source': SRC,
})

## Chart 3 — Inaugural: the “I” presidents and the “we” presidents (social + web)

Same as Chart 2 for inaugural addresses — left = 10 most “I” (% I), right = 10 most “we” (% we).

In [ ]:
render_targets(3, 'inaugural_i_vs_we_presidents', {
    'type': 'side_by_side_bars',
    'table_left': 'chart_inaug_top_i', 'table_right': 'chart_inaug_top_we',
    'category_col': 'president', 'value_col': 'pct', 'label_col': 'label',
    'left_title': 'Most “I”', 'right_title': 'Most “we”',
    'left_color': c('spice'), 'right_color': c('teal'),
    'title': 'Inaugural addresses — the “I” presidents and the “we” presidents',
    'subtitle': 'Left: share of first-person pronouns that are “I”. Right: share that are “we”. Top 10 each.',
    'source': SRC,
})

## Chart 4 — every SOTU president’s lean toward “I” or “we”, chronological (WEB ONLY)

All SOTU presidents (≥2 SOTUs) in chronological order, diverging from a 50/50 centerline:
right leans “I”, left leans “we”. Side headers name the direction once (not per bar);
tall canvas so the bars aren’t squished. Too tall for a social card.

In [ ]:
render_targets(4, 'sotu_lean_all_chronological', {
    'type': 'diverging_bars', 'table': 'chart_sotu_lean_all',
    'category_col': 'president', 'value_col': 'lean', 'label_col': 'label',
    'pos_color': c('spice'), 'neg_color': c('teal'), 'zero_label': '50/50', 'sort': False,
    'side_headers': {'neg': '◄ leans “we”', 'pos': 'leans “I” ►'},
    'inside_labels': True,
    'title': 'State of the Union — every president’s lean toward “I” or “we”',
    'subtitle': 'Chronological (top = earliest). Bar = distance from a 50/50 split; % is the leaning side\u2019s share. ≥2 SOTUs.',
    'source': SRC,
}, social=False, web=True, web_img_height=1800)

## Chart 5 — State of the Union: average length by president (WEB ONLY)

Straight average word count within the SOTU lens. The longest are pre-1913 *written*
messages; the shortest are *spoken* addresses — the caption carries that break. Tall canvas.

In [ ]:
render_targets(5, 'sotu_length_by_president', {
    'type': 'single_ranked_bars', 'table': 'chart_sotu_length',
    'category_col': 'president', 'value_col': 'avg_words', 'label_col': 'label',
    'bar_color': c('navy'),
    'title': 'State of the Union — average length by president (words)',
    'subtitle': 'Longest are pre-1913 WRITTEN messages; shortest are spoken addresses. ≥2 SOTUs.',
    'source': SRC,
}, social=False, web=True, web_img_height=1800)

## Chart 6 — speeches in the corpus per year in office (WEB ONLY)

Curated-corpus coverage rate: how many of a president’s speeches the Miller Center
*included* per year in office (n ÷ tenure), NOT their true speaking output. Tenures ≥1yr.
Tall canvas.

In [ ]:
render_targets(6, 'speeches_per_year_in_office', {
    'type': 'single_ranked_bars', 'table': 'chart_speeches_per_year',
    'category_col': 'president', 'value_col': 'speeches_per_year', 'label_col': 'label',
    'bar_color': c('gold'),
    'title': 'Speeches in the corpus per year in office',
    'subtitle': 'CORPUS COVERAGE rate (curated inclusion ÷ tenure), not total output. Tenures ≥1yr only.',
    'source': 'Miller Center (UVA) speech archive + curated term dates',
}, social=False, web=True, web_img_height=1800)

## Chart 7 — applause and laughter in presidential speeches (social + web)

Transcriber markers of live audience reaction per president (≥20 markers), applause vs
laughter — almost entirely a TV-era phenomenon. Footnote names the rare reactions (the
single boo, the lone cheers).

In [ ]:
render_targets(7, 'applause_and_laughter', {
    'type': 'stacked_bars', 'table': 'chart_stage_directions', 'category_col': 'president',
    'segments': [
        {'value_col': 'applause', 'label': 'Applause', 'color': c('teal'), 'text_color': '#E9D8A6'},
        {'value_col': 'laughter', 'label': 'Laughter', 'color': c('gold'), 'text_color': '#003049'},
    ],
    'title': 'Applause and laughter in presidential speeches',
    'subtitle': 'Transcriber markers of live audience reaction per president (≥20). Almost entirely a TV-era phenomenon.',
    'footnote': 'The only recorded boo: Donald Trump. The only cheers: Barack Obama and George H. W. Bush (1 each). Applause + laughter are 99%+ of all audience-reaction cues.',
    'source': SRC,
})

## Chart 8 — every president’s most-used word, 1789→present (social + web)

Each president’s single most-frequent word, chronological, bar length = that word’s count,
colored by word-family. Rendered a bit taller for social (1600×1080) so all rows fit while
staying social-postable; standard web canvas for the page.

In [ ]:
render_targets(8, 'top_word_timeline', {
    'type': 'single_ranked_bars', 'table': 'chart_top_word_timeline',
    'category_col': 'row_label', 'value_col': 'count', 'label_col': 'label',
    'color_col': 'color', 'sort': None,
    'legend': [{'label':'states/union','color':c('teal')},{'label':'world/peace','color':c('gold')},
               {'label':'people/country','color':c('spice')},{'label':'other','color':c('gray')}],
    'title': 'Every president’s most-used word, 1789→present',
    'subtitle': 'Top word per president (chronological). “States” dominates 1789–1909, then “world”, then “people”.',
    'source': SRC,
}, social_img_height=1080)

## Word clouds (social only) — the picker carries every president on the web side

One standout per view: **Lincoln** (distinctive / TF-IDF), **FDR** (most-used / raw
frequency), **Reagan** (distinctive phrases / bigrams). Clouds show only words the
president SPOKE — transcriber cues ([Applause]/[Laughter]) are excluded (they are Chart 7).
Web visitors browse all 45 presidents × 3 views in the interactive picker, so the clouds
are social-only here.

In [ ]:
lincoln = con.execute("""SELECT word, tfidf FROM distinctive_words_by_president
  WHERE president='Abraham Lincoln' ORDER BY tfidf DESC LIMIT 60""").df()
render_social_only(9, 'wc_lincoln_distinctive', {
    'type': 'word_cloud', 'table': lincoln, 'word_col': 'word', 'weight_col': 'tfidf',
    'title': 'Abraham Lincoln — distinctive words',
    'subtitle': 'Words he used far more than other presidents (TF-IDF). Spoken words only.',
    'source': SRC,
})

In [ ]:
fdr = con.execute("""SELECT word, count FROM word_freq_by_president
  WHERE president='Franklin D. Roosevelt' ORDER BY count DESC LIMIT 60""").df()
render_social_only(10, 'wc_fdr_mostused', {
    'type': 'word_cloud', 'table': fdr, 'word_col': 'word', 'weight_col': 'count',
    'title': 'Franklin D. Roosevelt — most-used words',
    'subtitle': 'His most-frequent spoken words (common function words removed).',
    'source': SRC,
})

In [ ]:
reagan = con.execute("""SELECT word, tfidf FROM distinctive_phrases_by_president
  WHERE president='Ronald Reagan' ORDER BY tfidf DESC LIMIT 60""").df()
render_social_only(11, 'wc_reagan_phrases', {
    'type': 'word_cloud', 'table': reagan, 'word_col': 'word', 'weight_col': 'tfidf',
    'title': 'Ronald Reagan — distinctive phrases',
    'subtitle': 'Two-word phrases he used far more than other presidents (TF-IDF). Spoken words only.',
    'source': SRC,
})

---
**Next:** independent Grok validation pass (report → `artifacts/`), then write
`scripts/validate_charts.py` (must exit 0) before release curation per `public-release.md`.

---
## Cleanup
Close the read-only DuckDB connection so the lock is released for other tools.

In [ ]:
con.close()
print('connection closed')